# Hitmakers — Full Model Pipeline & Comparison

Six ensemble/boosting models: AdaBoost Linear · AdaBoost Tree · Random Forest · LightGBM · XGBoost · CatBoost

Per-model pipeline:
1. Full-feature Optuna (penalized: AUC − λ×gap, λ=0.3)
2. Cross-validated permutation importance (leakage-safe)
3. Dynamic genre consolidation (keep genres with CV-perm-importance > 0)
4. Ablation study (centrality analysis embedded)
5. Forward selection
6. Re-tune Optuna for candidate n values (n_auc, n_gap)
7. λ robustness check (λ ∈ {0.1, 0.3, 0.5, 1.0}) + penalized auto-select n_optimal
8. Final model evaluation (ROC, confusion matrix, calibration, PR, PDP, lift curve)
9. OOF threshold tuning (leakage-safe — threshold chosen on training OOF predictions)

Cross-model section: metrics table, bar charts, lift overlay, disagreement prediction table, unsigned + signed feature importance heatmaps.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import (
    roc_auc_score, log_loss, brier_score_loss,
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score,
    ConfusionMatrixDisplay,
)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# LAM: penalty in Optuna objective (AUC - LAM * overfit_gap).
# Willing to sacrifice LAM units of AUC to reduce gap by 1.
# Robustness across lambda values is checked in Step 7.
LAM             = 0.3
N_SPLITS        = 5
N_TRIALS_FULL   = 30
N_TRIALS_RETUNE = 30
RANDOM_STATE    = 42

MODEL_NAMES = [
    'AdaBoost Linear',
    'AdaBoost Tree',
    'Random Forest',
    'LightGBM',
    'XGBoost',
    'CatBoost',
]

# SGDClassifier (AdaBoost Linear base) is scale-sensitive.
# Tree-based models are scale-invariant.
MODEL_NEEDS_SCALER = {name: (name == 'AdaBoost Linear') for name in MODEL_NAMES}

In [ ]:
# years_through_first_top_20_hit is stored as CSV index — reset_index() recovers it.
# 80/20 stratified split. No separate validation set needed:
# Optuna + 5-fold CV handles hyperparameter selection without a held-out val set,
# and our small dataset (759 artists) benefits from maximising training data.

df = pd.read_csv('df_artists_final.csv', index_col=0).reset_index()
X  = df.drop(columns=['top_20_hitmaker'])
y  = df['top_20_hitmaker']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print(f'Full dataset:  {df.shape}')
print(f'Train / Test:  {X_train.shape[0]} / {X_test.shape[0]}')
print(f'Features:      {X.shape[1]}')
print(f'Class balance (full):')
print(y.value_counts(normalize=True).round(3).rename({0.0: 'One-hit Wonder', 1.0: 'Hitmaker'}))

## EDA

In [ ]:
# Hitmaker rate per genre — shows which genres are over- or under-represented.
# Red bars: genres where hitmaker rate exceeds the overall mean.
# Motivates genre consolidation in Step 3: high-signal genres kept separately,
# low-signal genres merged into a single 'other' flag.

genre_cols   = [c for c in X.columns if c.startswith('artist_genre_')]
overall_mean = y.mean()
rates = []
for col in genre_cols:
    mask = df[col] == 1
    if mask.sum() == 0:
        continue
    rates.append({
        'Genre': col.replace('artist_genre_', ''),
        'Hitmaker Rate': df.loc[mask, 'top_20_hitmaker'].mean(),
        'N': mask.sum(),
    })

df_rates = pd.DataFrame(rates).sort_values('Hitmaker Rate', ascending=True)
fig, ax  = plt.subplots(figsize=(9, 7))
colors   = ['#d62728' if r > overall_mean else '#aec7e8' for r in df_rates['Hitmaker Rate']]
bars     = ax.barh(df_rates['Genre'], df_rates['Hitmaker Rate'], color=colors)
ax.axvline(overall_mean, color='black', linestyle='--', lw=1.5,
           label=f'Overall mean ({overall_mean:.2f})')
for bar, n in zip(bars, df_rates['N']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
            f'n={n}', va='center', fontsize=8)
ax.set_xlabel('Hitmaker Rate')
ax.set_title('Hitmaker Rate by Genre')
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Centrality distributions by class — shows whether network position
# (betweenness, eigenvector, harmonic closeness) differs between hitmakers
# and one-hit wonders. Computed on the training set only; the test set is
# kept completely unseen at this stage.
# Pearson correlations with target provide a quick linear signal check.

cent_cols = [c for c in X.columns if 'centrality' in c]
df_eda    = X_train.copy()
df_eda['top_20_hitmaker'] = y_train.values

fig, axes = plt.subplots(1, len(cent_cols), figsize=(5 * len(cent_cols), 5))
for ax, col in zip(axes, cent_cols):
    for label, grp in df_eda.groupby('top_20_hitmaker'):
        lbl = 'Hitmaker' if label == 1 else 'One-hit Wonder'
        ax.hist(grp[col].dropna(), bins=25, alpha=0.6, label=lbl, density=True)
    short = col.replace('_centrality_top20_rolling5', '').replace('_', ' ')
    ax.set_title(short, fontsize=10)
    ax.set_xlabel('Value'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.suptitle('Centrality Distribution by Class (Train Set)', fontsize=13)
plt.tight_layout(); plt.show()

print('Pearson correlation with top_20_hitmaker (train set):')
print(df_eda[cent_cols + ['top_20_hitmaker']].corr()['top_20_hitmaker']
      .drop('top_20_hitmaker').round(3))

## Model setup

In [ ]:
# Single entry point for constructing any of the 6 classifiers.
# All models are sklearn-compatible (fit / predict_proba).
# Key design choices:
#   AdaBoost Linear  — SGDClassifier(loss='log_loss') weak learner (linear, scale-sensitive).
#   AdaBoost Tree    — DecisionTreeClassifier weak learner (classic Freund & Schapire 1997).
#   Random Forest    — class_weight='balanced' for the 57/43 imbalance.
#   LightGBM/XGBoost/CatBoost — gradient boosting with regularization tuned by Optuna.

def build_model(name, params):
    if name == 'AdaBoost Linear':
        base = SGDClassifier(
            loss='log_loss',
            alpha=params.get('alpha', 1e-4),
            penalty=params.get('penalty', 'l2'),
            l1_ratio=params.get('l1_ratio', 0.15),
            max_iter=1000, random_state=RANDOM_STATE,
        )
        return AdaBoostClassifier(
            estimator=base,
            n_estimators=params['n_estimators'],
            learning_rate=params['learning_rate'],
            random_state=RANDOM_STATE,
        )
    elif name == 'AdaBoost Tree':
        base = DecisionTreeClassifier(
            max_depth=params.get('max_depth', 1),
            random_state=RANDOM_STATE,
        )
        return AdaBoostClassifier(
            estimator=base,
            n_estimators=params['n_estimators'],
            learning_rate=params['learning_rate'],
            random_state=RANDOM_STATE,
        )
    elif name == 'Random Forest':
        return RandomForestClassifier(
            n_estimators=params['n_estimators'],
            max_depth=params.get('max_depth', None),
            min_samples_leaf=params.get('min_samples_leaf', 1),
            max_features=params.get('max_features', 'sqrt'),
            class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1,
        )
    elif name == 'LightGBM':
        return LGBMClassifier(
            n_estimators=params['n_estimators'],
            learning_rate=params['learning_rate'],
            max_depth=params.get('max_depth', -1),
            num_leaves=params.get('num_leaves', 31),
            reg_alpha=params.get('reg_alpha', 0.0),
            reg_lambda=params.get('reg_lambda', 0.0),
            min_child_samples=params.get('min_child_samples', 20),
            random_state=RANDOM_STATE, verbose=-1, n_jobs=-1,
        )
    elif name == 'XGBoost':
        return XGBClassifier(
            n_estimators=params['n_estimators'],
            learning_rate=params['learning_rate'],
            max_depth=params.get('max_depth', 3),
            min_child_weight=params.get('min_child_weight', 1),
            gamma=params.get('gamma', 0.0),
            subsample=params.get('subsample', 0.8),
            colsample_bytree=params.get('colsample_bytree', 0.8),
            reg_alpha=params.get('reg_alpha', 0.0),
            reg_lambda=params.get('reg_lambda', 1.0),
            random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0,
        )
    elif name == 'CatBoost':
        return CatBoostClassifier(
            iterations=params['n_estimators'],
            learning_rate=params['learning_rate'],
            depth=params.get('depth', 4),
            l2_leaf_reg=params.get('l2_leaf_reg', 3.0),
            random_strength=params.get('random_strength', 1.0),
            border_count=params.get('border_count', 128),
            random_seed=RANDOM_STATE, verbose=0,
        )
    raise ValueError(f'Unknown model: {name}')

In [ ]:
# make_optuna_objective: returns a trial->penalized_score function per model.
#   Penalized score = CV AUC - LAM * overfit_gap.
#   try/except handles the rare AdaBoost 'base estimator worse than random' error.
#
# cv_evaluate: 5-fold evaluation returning mean +/- std across folds.
#   CV AUC std captures fold-to-fold variance -- a wide std signals instability.

def make_optuna_objective(name, X, y, lam, skf):
    def objective(trial):
        if name == 'AdaBoost Linear':
            penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet'])
            params = {
                'n_estimators':  trial.suggest_int('n_estimators', 50, 300),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 2.0, log=True),
                'alpha':         trial.suggest_float('alpha', 1e-5, 1.0, log=True),
                'penalty':       penalty,
                'l1_ratio':      trial.suggest_float('l1_ratio', 0.1, 0.9) if penalty == 'elasticnet' else 0.15,
            }
        elif name == 'AdaBoost Tree':
            params = {
                'n_estimators':  trial.suggest_int('n_estimators', 50, 300),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 2.0, log=True),
                'max_depth':     trial.suggest_int('max_depth', 1, 4),
            }
        elif name == 'Random Forest':
            params = {
                'n_estimators':     trial.suggest_int('n_estimators', 100, 500),
                'max_depth':        trial.suggest_int('max_depth', 2, 15),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
                'max_features':     trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5, 0.7]),
            }
        elif name == 'LightGBM':
            params = {
                'n_estimators':      trial.suggest_int('n_estimators', 50, 500),
                'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
                'max_depth':         trial.suggest_int('max_depth', 3, 12),
                'num_leaves':        trial.suggest_int('num_leaves', 8, 128),
                'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
                'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
            }
        elif name == 'XGBoost':
            params = {
                'n_estimators':     trial.suggest_int('n_estimators', 50, 500),
                'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
                'max_depth':        trial.suggest_int('max_depth', 2, 8),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
                'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
                'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
                'reg_alpha':        trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
                'reg_lambda':       trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
            }
        elif name == 'CatBoost':
            params = {
                'n_estimators':    trial.suggest_int('n_estimators', 50, 500),
                'learning_rate':   trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
                'depth':           trial.suggest_int('depth', 2, 8),
                'l2_leaf_reg':     trial.suggest_float('l2_leaf_reg', 0.5, 10.0, log=True),
                'random_strength': trial.suggest_float('random_strength', 0.1, 5.0, log=True),
                'border_count':    trial.suggest_int('border_count', 32, 255),
            }
        fold_val_auc, fold_train_auc = [], []
        for train_idx, val_idx in skf.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = build_model(name, params)
            try:
                model.fit(X_tr, y_tr)
                fold_val_auc.append(roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
                fold_train_auc.append(roc_auc_score(y_tr, model.predict_proba(X_tr)[:, 1]))
            except Exception:
                fold_val_auc.append(np.nan)
                fold_train_auc.append(np.nan)
        val_auc = np.nanmean(fold_val_auc)
        gap     = np.nanmean(fold_train_auc) - val_auc
        return val_auc - lam * gap
    return objective


def cv_evaluate(name, X, y, params, skf):
    fold_val_auc, fold_train_auc, fold_logloss, fold_brier = [], [], [], []
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        model = build_model(name, params)
        try:
            model.fit(X_tr, y_tr)
            proba    = model.predict_proba(X_val)[:, 1]
            proba_tr = model.predict_proba(X_tr)[:, 1]
            fold_val_auc.append(roc_auc_score(y_val, proba))
            fold_train_auc.append(roc_auc_score(y_tr, proba_tr))
            fold_logloss.append(log_loss(y_val, proba))
            fold_brier.append(brier_score_loss(y_val, proba))
        except Exception:
            fold_val_auc.append(np.nan); fold_train_auc.append(np.nan)
            fold_logloss.append(np.nan); fold_brier.append(np.nan)
    valid_val = [v for v in fold_val_auc if not np.isnan(v)]
    return {
        'CV AUC':      np.nanmean(fold_val_auc),
        'CV AUC Std':  np.std(valid_val) if valid_val else np.nan,
        'Train AUC':   np.nanmean(fold_train_auc),
        'Overfit Gap': np.nanmean(fold_train_auc) - np.nanmean(fold_val_auc),
        'Logloss':     np.nanmean(fold_logloss),
        'BrierScore':  np.nanmean(fold_brier),
    }

## Pipeline

In [ ]:
# Imputer fit on training set only -- prevents leakage of test-set statistics.
# StandardScaler applied only for AdaBoost Linear: SGDClassifier gradient updates
# depend on feature magnitude, while tree-based models split on rank order and
# are invariant to monotonic transformations like scaling.
# Both imputer and scaler are stored in PIPE so they can be reapplied to any
# feature subset derived during forward selection.

PIPE = {name: {} for name in MODEL_NAMES}

for name in MODEL_NAMES:
    imputer  = SimpleImputer(strategy='median')
    X_tr_imp = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns, index=X_train.index,
    )
    X_te_imp = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns, index=X_test.index,
    )
    if MODEL_NEEDS_SCALER[name]:
        scaler   = StandardScaler()
        X_tr_imp = pd.DataFrame(
            scaler.fit_transform(X_tr_imp),
            columns=X_tr_imp.columns, index=X_tr_imp.index,
        )
        X_te_imp = pd.DataFrame(
            scaler.transform(X_te_imp),
            columns=X_te_imp.columns, index=X_te_imp.index,
        )
        PIPE[name]['scaler'] = scaler
    PIPE[name]['imputer']       = imputer
    PIPE[name]['X_train_clean'] = X_tr_imp
    PIPE[name]['X_test_clean']  = X_te_imp

print('Preprocessing complete.')
for name in MODEL_NAMES:
    print(f'  {name:20s}  train {PIPE[name]["X_train_clean"].shape}'
          f'  NaN={PIPE[name]["X_train_clean"].isna().sum().sum()}'
          f'  scaled={MODEL_NEEDS_SCALER[name]}')

### Step 1 — Full-feature Optuna tuning

In [ ]:
# Tune each model on the full 26-feature set before any feature selection.
# We tune first so that permutation importance in Step 2 reflects a
# well-regularized model rather than arbitrary default hyperparameters.
# Objective: AUC - LAM * overfit_gap (LAM=0.3).
# Expected runtime: ~20-40 min total across all 6 models.

for name in MODEL_NAMES:
    print(f'\n{"="*60}')
    print(f'  Optuna (full features): {name}  [{N_TRIALS_FULL} trials]')
    print(f'{"="*60}')
    X_tr  = PIPE[name]['X_train_clean']
    study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
    study.optimize(
        make_optuna_objective(name, X_tr, y_train, LAM, skf),
        n_trials=N_TRIALS_FULL, show_progress_bar=True,
    )
    res = cv_evaluate(name, X_tr, y_train, study.best_params, skf)
    PIPE[name]['study_full']  = study
    PIPE[name]['params_full'] = study.best_params
    PIPE[name]['cv_full']     = res
    print(f'  CV AUC:      {res["CV AUC"]:.4f} +/- {res["CV AUC Std"]:.4f}')
    print(f'  Train AUC:   {res["Train AUC"]:.4f}')
    print(f'  Overfit Gap: {res["Overfit Gap"]:.4f}')
    print(f'  Best params: {study.best_params}')

print('\n\n-- Full-Feature Optuna Summary --')
rows = []
for name in MODEL_NAMES:
    r = PIPE[name]['cv_full']
    rows.append({
        'Model':       name,
        'CV AUC':      f'{r["CV AUC"]:.4f} +/- {r["CV AUC Std"]:.4f}',
        'Train AUC':   round(r['Train AUC'], 4),
        'Overfit Gap': round(r['Overfit Gap'], 4),
        'Logloss':     round(r['Logloss'], 4),
        'BrierScore':  round(r['BrierScore'], 4),
    })
print(pd.DataFrame(rows).set_index('Model').to_string())

### Step 2 — Cross-validated permutation importance

In [ ]:
# Standard permutation importance is computed on data the model trained on,
# which is slightly optimistic (the model has memorized some of that data).
# Here we compute importance on the *validation* fold of each CV split:
# the model never saw that fold, so scores reflect genuine out-of-sample
# feature contributions. Final importance = mean across folds;
# std = fold-to-fold variability (wide std = unstable importance).
# This ranking is used to order features for forward selection in Step 5.

for name in MODEL_NAMES:
    print(f'\n{name}')
    X_tr   = PIPE[name]['X_train_clean']
    params = PIPE[name]['params_full']
    fold_importances = []
    for train_idx, val_idx in skf.split(X_tr, y_train):
        X_f, X_v = X_tr.iloc[train_idx], X_tr.iloc[val_idx]
        y_f, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model = build_model(name, params)
        try:
            model.fit(X_f, y_f)
            perm = permutation_importance(
                model, X_v, y_v,
                n_repeats=5, random_state=RANDOM_STATE, scoring='roc_auc',
            )
            fold_importances.append(perm.importances_mean)
        except Exception:
            pass
    mean_imp = np.mean(fold_importances, axis=0)
    std_imp  = np.std(fold_importances, axis=0)
    perm_df  = pd.DataFrame({
        'Feature':    X_tr.columns,
        'Importance': mean_imp,
        'Std':        std_imp,
    }).sort_values('Importance', ascending=False).reset_index(drop=True)
    PIPE[name]['perm_full'] = perm_df
    print(perm_df.to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(20, 16))
for ax, name in zip(axes.flat, MODEL_NAMES):
    df_p   = PIPE[name]['perm_full'].sort_values('Importance', ascending=True)
    colors = ['#d62728' if v > 0 else '#aec7e8' for v in df_p['Importance']]
    ax.barh(df_p['Feature'], df_p['Importance'], xerr=df_p['Std'],
            color=colors, error_kw={'ecolor': 'gray', 'capsize': 2})
    ax.axvline(0, color='black', linewidth=1)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Mean CV AUC Drop')
    ax.tick_params(axis='y', labelsize=7)
plt.suptitle('Cross-Validated Permutation Importance -- Full Feature Set', fontsize=14)
plt.tight_layout(); plt.show()

### Step 3 — Dynamic genre consolidation

In [ ]:
# Genre dummies with CV permutation importance <= 0 are merged into a single
# 'artist_genre_other' binary flag (1 if the artist belongs to any low-signal genre).
# High-signal genres (importance > 0) are kept as separate dummies.
# Done independently per model: the same genre may be informative for CatBoost
# but noise for AdaBoost Linear -- no hardcoded genre lists anywhere.
# Consolidation derived from training data only and applied to both train and test.

for name in MODEL_NAMES:
    X_tr    = PIPE[name]['X_train_clean']
    X_te    = PIPE[name]['X_test_clean']
    perm_df = PIPE[name]['perm_full']

    all_genre_cols = [c for c in X_tr.columns if c.startswith('artist_genre_')]
    genre_imp      = perm_df.set_index('Feature')['Importance']
    high_signal    = [c for c in all_genre_cols if genre_imp.get(c, 0) > 0]
    low_signal     = [c for c in all_genre_cols if c not in high_signal]

    X_tr_cons = X_tr.drop(columns=low_signal).copy()
    X_te_cons = X_te.drop(columns=low_signal).copy()
    if low_signal:
        X_tr_cons['artist_genre_other'] = (X_tr[low_signal].sum(axis=1) > 0).astype(int)
        X_te_cons['artist_genre_other'] = (X_te[low_signal].sum(axis=1) > 0).astype(int)

    PIPE[name]['X_train_cons']       = X_tr_cons
    PIPE[name]['X_test_cons']        = X_te_cons
    PIPE[name]['high_signal_genres'] = high_signal
    PIPE[name]['low_signal_genres']  = low_signal

    print(f'{name}:')
    print(f'  kept {len(high_signal)} genres: {[c.replace("artist_genre_", "") for c in high_signal]}')
    print(f'  merged {len(low_signal)} -> artist_genre_other  |  total features: {X_tr_cons.shape[1]}')

### Step 4 — Ablation study

In [ ]:
# Drop one feature at a time from the consolidated set and measure the change
# in CV AUC relative to the full consolidated baseline.
# Features with Delta_AUC >= 0 when dropped are noise candidates -- removing them
# does not hurt (or even helps) performance.
# Centrality features are explicitly highlighted: models may disagree on whether
# network position helps (e.g. harmonic_closeness helped CatBoost but hurt
# AdaBoost Tree in previous notebooks -- the per-model ablation captures this).

def ablation_cv(name, X, y, params, skf):
    fold_val, fold_tr = [], []
    for train_idx, val_idx in skf.split(X, y):
        X_f, X_v = X.iloc[train_idx], X.iloc[val_idx]
        y_f, y_v = y.iloc[train_idx], y.iloc[val_idx]
        model = build_model(name, params)
        try:
            model.fit(X_f, y_f)
            fold_val.append(roc_auc_score(y_v, model.predict_proba(X_v)[:, 1]))
            fold_tr.append(roc_auc_score(y_f, model.predict_proba(X_f)[:, 1]))
        except Exception:
            fold_val.append(np.nan); fold_tr.append(np.nan)
    val_auc = np.nanmean(fold_val)
    return val_auc, np.nanmean(fold_tr) - val_auc

for name in MODEL_NAMES:
    print(f'\n{"─"*60}\nAblation: {name}')
    X_tr   = PIPE[name]['X_train_cons']
    params = PIPE[name]['params_full']
    baseline_auc, baseline_gap = ablation_cv(name, X_tr, y_train, params, skf)
    print(f'  Baseline: AUC={baseline_auc:.4f}  Gap={baseline_gap:.4f}')
    rows = []
    for feat in X_tr.columns:
        auc, gap = ablation_cv(name, X_tr.drop(columns=[feat]), y_train, params, skf)
        rows.append({'Feature': feat, 'CV AUC': auc,
                     'Delta AUC': auc - baseline_auc, 'Gap': gap})
    df_abl = pd.DataFrame(rows).sort_values('Delta AUC', ascending=False)
    PIPE[name]['ablation'] = df_abl
    cent_cols = [c for c in X_tr.columns if 'centrality' in c]
    print('  Centrality (Delta AUC when dropped):')
    for _, row in df_abl[df_abl['Feature'].isin(cent_cols)].iterrows():
        print(f'    {row["Feature"][:50]:50s}  Delta={row["Delta AUC"]:+.4f}')

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
for ax, name in zip(axes.flat, MODEL_NAMES):
    df_abl = PIPE[name]['ablation'].sort_values('Delta AUC', ascending=True)
    colors = ['#2ca02c' if v > 0 else '#d62728' for v in df_abl['Delta AUC']]
    ax.barh(df_abl['Feature'], df_abl['Delta AUC'], color=colors)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('DeltaCV AUC (positive = dropping helps)')
    ax.tick_params(axis='y', labelsize=7)
plt.suptitle('Ablation Study -- Consolidated Feature Set', fontsize=14)
plt.tight_layout(); plt.show()

### Step 5 — Forward selection

In [ ]:
# Cross-validated permutation importance is recomputed on the consolidated feature
# set (post-genre-consolidation) to get the feature ranking for this reduced space.
# Features are added greedily in order of importance. We track CV AUC, overfit gap,
# Logloss, and Brier score at each step to find the optimal feature count.
# Two candidates are passed to the re-tune step:
#   n_auc: feature count that maximises CV AUC
#   n_gap: feature count that minimises overfit gap

for name in MODEL_NAMES:
    print(f'\n{"─"*60}\nForward selection: {name}')
    X_tr   = PIPE[name]['X_train_cons']
    params = PIPE[name]['params_full']

    # CV permutation importance on consolidated set
    fold_importances = []
    for train_idx, val_idx in skf.split(X_tr, y_train):
        X_f, X_v = X_tr.iloc[train_idx], X_tr.iloc[val_idx]
        y_f, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model = build_model(name, params)
        try:
            model.fit(X_f, y_f)
            perm = permutation_importance(
                model, X_v, y_v,
                n_repeats=5, random_state=RANDOM_STATE, scoring='roc_auc',
            )
            fold_importances.append(perm.importances_mean)
        except Exception:
            pass
    mean_imp     = np.mean(fold_importances, axis=0)
    perm_cons_df = pd.DataFrame({
        'Feature': X_tr.columns, 'Importance': mean_imp,
    }).sort_values('Importance', ascending=False).reset_index(drop=True)
    feature_order = perm_cons_df['Feature'].tolist()
    PIPE[name]['perm_cons']     = perm_cons_df
    PIPE[name]['feature_order'] = feature_order

    # Forward selection
    sel_results = []
    start_n = min(3, len(feature_order))
    for n_feats in range(start_n, len(feature_order) + 1):
        feats = feature_order[:n_feats]
        X_sub = X_tr[feats]
        fold_val, fold_tr, fold_ll, fold_bs = [], [], [], []
        for train_idx, val_idx in skf.split(X_sub, y_train):
            X_f, X_v = X_sub.iloc[train_idx], X_sub.iloc[val_idx]
            y_f, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]
            model = build_model(name, params)
            try:
                model.fit(X_f, y_f)
                p_v = model.predict_proba(X_v)[:, 1]
                p_t = model.predict_proba(X_f)[:, 1]
                fold_val.append(roc_auc_score(y_v, p_v))
                fold_tr.append(roc_auc_score(y_f, p_t))
                fold_ll.append(log_loss(y_v, p_v))
                fold_bs.append(brier_score_loss(y_v, p_v))
            except Exception:
                fold_val.append(np.nan); fold_tr.append(np.nan)
                fold_ll.append(np.nan);  fold_bs.append(np.nan)
        val_auc = np.nanmean(fold_val); tr_auc = np.nanmean(fold_tr)
        sel_results.append({
            'n_features':  n_feats,
            'CV AUC':      val_auc,
            'Train AUC':   tr_auc,
            'Overfit Gap': tr_auc - val_auc,
            'Logloss':     np.nanmean(fold_ll),
            'BrierScore':  np.nanmean(fold_bs),
        })
        print(f'  n={n_feats:2d} +[{feature_order[n_feats-1][:40]:40s}]  '
              f'AUC={val_auc:.4f}  Gap={tr_auc - val_auc:.4f}')
    df_sel = pd.DataFrame(sel_results).set_index('n_features')
    PIPE[name]['forward_sel'] = df_sel

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for ax, name in zip(axes.flat, MODEL_NAMES):
    df_s   = PIPE[name]['forward_sel']
    best_n = df_s['CV AUC'].idxmax()
    ax.plot(df_s.index, df_s['CV AUC'],      'o-', color='steelblue', lw=2, label='CV AUC')
    ax.plot(df_s.index, df_s['Overfit Gap'], 's--', color='#d62728', lw=1.5, label='Overfit Gap')
    ax.axvline(best_n, color='black', linestyle=':', lw=1.5)
    ax.text(best_n + 0.1, df_s['CV AUC'].max(), f'n={best_n}', fontsize=9)
    ax.set_title(name, fontsize=11); ax.set_xlabel('Number of Features')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.suptitle('Forward Selection -- CV AUC and Overfit Gap', fontsize=14)
plt.tight_layout(); plt.show()

### Step 6 — Re-tune Optuna on candidate feature counts

In [ ]:
# Hyperparameters tuned on 26 features may not be optimal for a trimmed subset.
# We re-tune for each candidate (n_auc, n_gap) separately, letting the search
# adapt to the new feature space -- e.g. a smaller set may need fewer estimators
# or different regularization strength.
# If n_auc == n_gap, only one Optuna study is run.

for name in MODEL_NAMES:
    print(f'\n{"="*60}\nRe-tune: {name}')
    df_sel    = PIPE[name]['forward_sel']
    feat_ord  = PIPE[name]['feature_order']
    X_tr_cons = PIPE[name]['X_train_cons']
    X_te_cons = PIPE[name]['X_test_cons']

    n_auc = int(df_sel['CV AUC'].idxmax())
    n_gap = int(df_sel['Overfit Gap'].idxmin())
    candidate_ns = sorted(set([n_auc, n_gap]))
    print(f'  n_auc={n_auc}, n_gap={n_gap}, candidates: {candidate_ns}')

    best_params_by_n = {}
    cv_results_by_n  = {}
    X_train_by_n     = {n: X_tr_cons[feat_ord[:n]] for n in candidate_ns}
    X_test_by_n      = {n: X_te_cons[feat_ord[:n]] for n in candidate_ns}

    for n in candidate_ns:
        print(f'\n  -- n={n}: {feat_ord[:n]}')
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
        study.optimize(
            make_optuna_objective(name, X_train_by_n[n], y_train, LAM, skf),
            n_trials=N_TRIALS_RETUNE, show_progress_bar=True,
        )
        best_params_by_n[n] = study.best_params
        cv_results_by_n[n]  = cv_evaluate(name, X_train_by_n[n], y_train, study.best_params, skf)
        res = cv_results_by_n[n]
        print(f'  CV AUC={res["CV AUC"]:.4f} +/- {res["CV AUC Std"]:.4f}  '
              f'Gap={res["Overfit Gap"]:.4f}  Logloss={res["Logloss"]:.4f}')

    PIPE[name]['best_params_by_n'] = best_params_by_n
    PIPE[name]['cv_results_by_n']  = cv_results_by_n
    PIPE[name]['X_train_by_n']     = X_train_by_n
    PIPE[name]['X_test_by_n']      = X_test_by_n

### Step 7 — Lambda robustness check + auto-select n_optimal

In [ ]:
# The penalized score (CV AUC - lambda * gap) selects which candidate n to use.
# lambda = 0.3 is our primary choice (LAM constant), but we check robustness across
# lambda in {0.1, 0.3, 0.5, 1.0}:
#   - Same n_optimal for all lambda -> choice is robust, lambda=0.3 is fine.
#   - n_optimal flips -> flag it and note which lambda was decisive.
# The final selected n, feature set, and params are stored in PIPE.

LAMBDA_GRID = [0.1, 0.3, 0.5, 1.0]

header = f'{"Model":<22}  {"n_auc":>5}  {"n_gap":>5}'
for lam_c in LAMBDA_GRID:
    header += f'  lam={lam_c}->n'
header += f'  {"n_opt":>5}  {"robust?":>8}'
print(header)
print('-' * 90)

for name in MODEL_NAMES:
    df_sel       = PIPE[name]['forward_sel']
    cv_res       = PIPE[name]['cv_results_by_n']
    feat_ord     = PIPE[name]['feature_order']
    X_train_by_n = PIPE[name]['X_train_by_n']
    X_test_by_n  = PIPE[name]['X_test_by_n']
    bp_by_n      = PIPE[name]['best_params_by_n']
    candidate_ns = sorted(cv_res.keys())

    df_cv = pd.DataFrame(cv_res).T.round(4)
    df_cv.index.name = 'n_features'
    n_auc = int(df_sel['CV AUC'].idxmax())
    n_gap = int(df_sel['Overfit Gap'].idxmin())

    lam_selections = {}
    for lam_c in LAMBDA_GRID:
        df_cv['_pen'] = df_cv['CV AUC'] - lam_c * df_cv['Overfit Gap']
        lam_selections[lam_c] = int(df_cv['_pen'].idxmax())
    df_cv.drop(columns=['_pen'], inplace=True)

    n_optimal = lam_selections[LAM]
    robust    = len(set(lam_selections.values())) == 1

    df_cv['penalized'] = df_cv['CV AUC'] - LAM * df_cv['Overfit Gap']
    PIPE[name]['df_cv_summary']  = df_cv
    PIPE[name]['n_optimal']      = n_optimal
    PIPE[name]['X_train_final']  = X_train_by_n[n_optimal]
    PIPE[name]['X_test_final']   = X_test_by_n[n_optimal]
    PIPE[name]['params_final']   = bp_by_n[n_optimal]
    PIPE[name]['lambda_robust']  = robust

    row = f'{name:<22}  {n_auc:>5}  {n_gap:>5}'
    for lam_c in LAMBDA_GRID:
        row += f'  {lam_selections[lam_c]:>10}'
    row += f'  {n_optimal:>5}  {"ok" if robust else "check":>8}'
    print(row)

print('\n\nCV summaries at lambda=0.3:')
for name in MODEL_NAMES:
    n = PIPE[name]['n_optimal']
    r = PIPE[name]['cv_results_by_n'][n]
    print(f'  {name}: n_optimal={n}  '
          f'CV AUC={r["CV AUC"]:.4f} +/- {r["CV AUC Std"]:.4f}  '
          f'Gap={r["Overfit Gap"]:.4f}')

### Step 8 — Final model evaluation

In [ ]:
# Fit each final model on the full training set (n_optimal features, tuned params).
# Evaluate once on the held-out test set -- this is the only time the test set
# is used for reporting, ensuring an unbiased estimate of generalization performance.
# Produces per-model: ROC curve, confusion matrix, calibration curve, PR curve.
# Also computes permutation importance on the training set for the final feature set
# (used in the cross-model heatmap and for feature direction analysis).

def compute_lift(y_true, y_proba):
    """Cumulative gain and lift at each probability rank threshold."""
    n         = len(y_true)
    total_pos = np.sum(y_true)
    sorted_idx = np.argsort(y_proba)[::-1]
    y_sorted   = np.array(y_true)[sorted_idx]
    cum_pos    = np.cumsum(y_sorted)
    pop_frac   = np.arange(1, n + 1) / n
    gain       = cum_pos / total_pos
    lift       = gain / pop_frac
    return pop_frac, gain, lift

for name in MODEL_NAMES:
    print(f'\n{"="*60}')
    print(f'  FINAL MODEL: {name}  (n={PIPE[name]["n_optimal"]} features)')
    print(f'{"="*60}')
    X_tr   = PIPE[name]['X_train_final']
    X_te   = PIPE[name]['X_test_final']
    params = PIPE[name]['params_final']

    model = build_model(name, params)
    model.fit(X_tr, y_train)
    PIPE[name]['model_final'] = model

    y_proba_te = model.predict_proba(X_te)[:, 1]
    y_proba_tr = model.predict_proba(X_tr)[:, 1]
    y_pred_50  = (y_proba_te >= 0.5).astype(int)

    PIPE[name]['y_proba_te'] = y_proba_te
    PIPE[name]['y_proba_tr'] = y_proba_tr
    PIPE[name]['test_auc']   = roc_auc_score(y_test, y_proba_te)
    PIPE[name]['train_auc']  = roc_auc_score(y_train, y_proba_tr)
    PIPE[name]['gap']        = PIPE[name]['train_auc'] - PIPE[name]['test_auc']
    PIPE[name]['logloss']    = log_loss(y_test, y_proba_te)
    PIPE[name]['brier']      = brier_score_loss(y_test, y_proba_te)

    print(f'  Test AUC: {PIPE[name]["test_auc"]:.4f}  Train AUC: {PIPE[name]["train_auc"]:.4f}  Gap: {PIPE[name]["gap"]:.4f}')
    print(f'  Log Loss: {PIPE[name]["logloss"]:.4f}  Brier: {PIPE[name]["brier"]:.4f}')
    print(classification_report(y_test, y_pred_50, target_names=['One-hit Wonder', 'Hitmaker']))

    cm = confusion_matrix(y_test, y_pred_50)
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    fpr, tpr, _ = roc_curve(y_test, y_proba_te)
    axes[0,0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC={PIPE[name]["test_auc"]:.4f}')
    axes[0,0].plot([0,1],[0,1],'k--',lw=1)
    axes[0,0].set_xlabel('FPR'); axes[0,0].set_ylabel('TPR')
    axes[0,0].set_title('ROC Curve'); axes[0,0].legend(); axes[0,0].grid(True)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,1],
                xticklabels=['Non-hitmaker','Hitmaker'],
                yticklabels=['Non-hitmaker','Hitmaker'])
    axes[0,1].set_title('Confusion Matrix (threshold=0.5)')

    frac_pos, mean_pred = calibration_curve(y_test, y_proba_te, n_bins=10)
    axes[1,0].plot(mean_pred, frac_pos, 's-', color='steelblue', label=name)
    axes[1,0].plot([0,1],[0,1],'k--',lw=1.5,label='Perfect')
    axes[1,0].set_xlabel('Mean Predicted Probability'); axes[1,0].set_ylabel('Fraction Positive')
    axes[1,0].set_title('Calibration Curve'); axes[1,0].legend(); axes[1,0].grid(True)

    prec_v, rec_v, _ = precision_recall_curve(y_test, y_proba_te)
    ap = average_precision_score(y_test, y_proba_te)
    axes[1,1].plot(rec_v, prec_v, color='steelblue', lw=2, label=f'AP={ap:.4f}')
    axes[1,1].set_xlabel('Recall'); axes[1,1].set_ylabel('Precision')
    axes[1,1].set_title('Precision-Recall Curve'); axes[1,1].legend(); axes[1,1].grid(True)

    plt.suptitle(f'{name} -- Test Set Evaluation (n={PIPE[name]["n_optimal"]})', fontsize=13)
    plt.tight_layout(); plt.show()

    # Permutation importance on final feature set
    perm_fin = permutation_importance(
        model, X_tr, y_train,
        n_repeats=20, random_state=RANDOM_STATE, scoring='roc_auc',
    )
    perm_fin_df = pd.DataFrame({
        'Feature':    X_tr.columns,
        'Importance': perm_fin.importances_mean,
        'Std':        perm_fin.importances_std,
    }).sort_values('Importance', ascending=False).reset_index(drop=True)
    PIPE[name]['perm_final'] = perm_fin_df

    fig, ax = plt.subplots(figsize=(9, max(4, len(X_tr.columns) * 0.4)))
    df_p   = perm_fin_df.sort_values('Importance', ascending=True)
    colors = ['#d62728' if v > 0 else '#aec7e8' for v in df_p['Importance']]
    ax.barh(df_p['Feature'], df_p['Importance'], xerr=df_p['Std'],
            color=colors, error_kw={'ecolor': 'gray', 'capsize': 3})
    ax.axvline(0, color='black', lw=1)
    ax.set_xlabel('Mean AUC Drop')
    ax.set_title(f'{name} -- Final Feature Importance (n={PIPE[name]["n_optimal"]})')
    plt.tight_layout(); plt.show()

    # Feature direction table
    hm = y_train == 1
    dir_df = pd.DataFrame({
        'Perm Importance':      perm_fin_df.set_index('Feature')['Importance'],
        'Mean (hitmakers)':     X_tr[hm].mean(),
        'Mean (non-hitmakers)': X_tr[~hm].mean(),
    }).sort_values('Perm Importance', ascending=False).round(4)
    dir_df['Direction'] = (dir_df['Mean (hitmakers)'] > dir_df['Mean (non-hitmakers)']).map(
        {True: 'up -> Hitmaker', False: 'down -> Hitmaker'}
    )
    PIPE[name]['direction_df'] = dir_df
    print('\nFeature direction:')
    print(dir_df.to_string())

### Step 9 — Partial Dependence Plots

In [ ]:
# PDPs show how predicted hitmaker probability changes as a single feature varies,
# while all other features are held at their average (marginal) values.
# This translates model behavior into plain language for stakeholders:
# 'as charting songs increases from 1 to 15, hitmaker probability rises from X to Y.'
# We plot the top 3 features by permutation importance for each model.
# Note: for AdaBoost Linear, x-axis values are on the StandardScaler scale
# (mean=0, std=1) rather than original units -- interpret directions, not magnitudes.

for name in MODEL_NAMES:
    model     = PIPE[name]['model_final']
    X_tr      = PIPE[name]['X_train_final']
    top_feats = PIPE[name]['perm_final']['Feature'].head(3).tolist()
    scale_note = ' (scaled units)' if MODEL_NEEDS_SCALER[name] else ''

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    try:
        PartialDependenceDisplay.from_estimator(
            model, X_tr, features=top_feats,
            ax=axes, kind='average', grid_resolution=50,
            line_kw={'color': 'steelblue', 'lw': 2},
        )
        plt.suptitle(f'{name} -- Partial Dependence Plots (top 3 features{scale_note})', fontsize=13)
        plt.tight_layout(); plt.show()
    except Exception as e:
        print(f'{name}: PDP failed -- {e}')
        plt.close()

### Step 10 — OOF threshold tuning

In [ ]:
# Choosing the threshold by looking at test-set metrics is data leakage --
# the test set should be touched exactly once for final reporting.
# Instead, we generate out-of-fold (OOF) predictions on the training set:
# each fold's validation predictions come from a model that never saw that fold.
# We tune the threshold on OOF predictions, then apply to the test set only for
# final reporting.
# Fallback: if F1-peak threshold gives precision < 0.60, find the highest-F1
# threshold where precision >= 0.60 (avoids flagging too many false positives).

for name in MODEL_NAMES:
    X_tr   = PIPE[name]['X_train_final']
    params = PIPE[name]['params_final']

    oof_proba = np.zeros(len(y_train))
    for train_idx, val_idx in skf.split(X_tr, y_train):
        X_f, X_v = X_tr.iloc[train_idx], X_tr.iloc[val_idx]
        y_f, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model = build_model(name, params)
        try:
            model.fit(X_f, y_f)
            oof_proba[val_idx] = model.predict_proba(X_v)[:, 1]
        except Exception:
            oof_proba[val_idx] = 0.5

    thresholds = np.arange(0.05, 0.95, 0.01)
    precs = np.array([precision_score(y_train, (oof_proba >= t).astype(int), zero_division=0) for t in thresholds])
    recs  = np.array([recall_score(y_train,    (oof_proba >= t).astype(int), zero_division=0) for t in thresholds])
    f1s   = np.array([f1_score(y_train,        (oof_proba >= t).astype(int), zero_division=0) for t in thresholds])

    best_idx = np.argmax(f1s)
    if precs[best_idx] < 0.60:
        valid_mask = precs >= 0.60
        best_idx   = np.argmax(np.where(valid_mask, f1s, 0)) if valid_mask.any() else np.argmin(np.abs(thresholds - 0.5))

    chosen_thresh = round(thresholds[best_idx], 2)
    PIPE[name]['threshold']  = chosen_thresh
    PIPE[name]['oof_proba']  = oof_proba

    y_proba_te = PIPE[name]['y_proba_te']
    y_pred_t   = (y_proba_te >= chosen_thresh).astype(int)
    PIPE[name]['precision_final'] = precision_score(y_test, y_pred_t)
    PIPE[name]['recall_final']    = recall_score(y_test, y_pred_t)
    PIPE[name]['f1_final']        = f1_score(y_test, y_pred_t)

    print(f'{name}: threshold={chosen_thresh}  '
          f'P={PIPE[name]["precision_final"]:.3f}  '
          f'R={PIPE[name]["recall_final"]:.3f}  '
          f'F1={PIPE[name]["f1_final"]:.3f}')

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for ax, name in zip(axes.flat, MODEL_NAMES):
    oof_p = PIPE[name]['oof_proba']
    ts    = np.arange(0.05, 0.95, 0.01)
    ps = [precision_score(y_train, (oof_p >= t).astype(int), zero_division=0) for t in ts]
    rs = [recall_score(y_train,    (oof_p >= t).astype(int), zero_division=0) for t in ts]
    fs = [f1_score(y_train,        (oof_p >= t).astype(int), zero_division=0) for t in ts]
    ax.plot(ts, ps, color='steelblue',  lw=2, label='Precision')
    ax.plot(ts, rs, color='darkorange', lw=2, label='Recall')
    ax.plot(ts, fs, color='green',      lw=2, label='F1')
    ax.axvline(0.5,                     color='gray',  linestyle='--', lw=1, alpha=0.5, label='Default 0.5')
    ax.axvline(PIPE[name]['threshold'], color='green', linestyle=':',  lw=2,
               label=f'Chosen ({PIPE[name]["threshold"]})')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Threshold (on OOF predictions, training set)')
    ax.set_ylabel('Score')
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.05)
plt.suptitle('Threshold Tuning -- OOF Predictions (Training Set)', fontsize=14)
plt.tight_layout(); plt.show()

### Step 11 — Lift curves and prediction tables

In [ ]:
# Lift curve: at each percentile of the ranked probability list, how many times
# better are we than random at identifying hitmakers?
# 'Screening the top 30% of artists flagged by the model captures X% of all
# hitmakers -- Y times better than randomly selecting 30%.'
# This is the most intuitive metric for stakeholders making resource allocation
# decisions (e.g. which artists to invest in or sign).
#
# Prediction table: full test-set ranking with predicted probability, true label,
# and error type. Sorted by predicted probability descending.
# Useful for spotting systematic errors and for presentation moments.

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
for ax, name in zip(axes.flat, MODEL_NAMES):
    y_p = PIPE[name]['y_proba_te']
    pop_frac, gain, _ = compute_lift(y_test.values, y_p)
    ax.plot(pop_frac * 100, gain * 100, color='steelblue', lw=2, label=name)
    ax.plot([0, 100], [0, 100], 'k--', lw=1, label='Random')
    ax.fill_between(pop_frac * 100, gain * 100, pop_frac * 100, alpha=0.15, color='steelblue')
    ax.set_xlabel('% of Artists Screened (ranked by predicted probability)')
    ax.set_ylabel('% of Hitmakers Captured')
    ax.set_title(name, fontsize=11)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 100); ax.set_ylim(0, 105)
plt.suptitle('Cumulative Gain (Lift) Curves -- Individual Models', fontsize=14)
plt.tight_layout(); plt.show()

for name in MODEL_NAMES:
    thresh = PIPE[name]['threshold']
    y_p    = PIPE[name]['y_proba_te']
    y_pred = (y_p >= thresh).astype(int)
    pred_table = pd.DataFrame({
        'True Label':      y_test.values,
        'Predicted Prob':  y_p.round(3),
        'Predicted Label': y_pred,
        'Correct':         (y_test.values == y_pred).astype(int),
        'Error Type':      np.where(y_test.values == y_pred, 'Correct',
                           np.where((y_test.values == 0) & (y_pred == 1),
                                    'FP (false alarm)', 'FN (missed hitmaker)')),
    }, index=y_test.index).sort_values('Predicted Prob', ascending=False).reset_index()
    PIPE[name]['pred_table'] = pred_table
    acc = pred_table['Correct'].mean()
    fp  = (pred_table['Error Type'] == 'FP (false alarm)').sum()
    fn  = (pred_table['Error Type'] == 'FN (missed hitmaker)').sum()
    print(f'\n-- {name} (threshold={thresh}) -- Accuracy={acc:.3f}  FP={fp}  FN={fn} --')
    print(pred_table[['index','True Label','Predicted Prob','Predicted Label','Error Type']]
          .head(20).to_string(index=False))

## Cross-model comparison

In [ ]:
# All models compared on the same held-out test set with their OOF-tuned thresholds.
# CV AUC reported as mean +/- std from the re-tune step at n_optimal.
# This is the main results table for the presentation.

rows = []
for name in MODEL_NAMES:
    n = PIPE[name]['n_optimal']
    r = PIPE[name]['cv_results_by_n'][n]
    rows.append({
        'Model':       name,
        'N Features':  n,
        'Threshold':   PIPE[name]['threshold'],
        'CV AUC':      f'{r["CV AUC"]:.4f} +/- {r["CV AUC Std"]:.4f}',
        'Test AUC':    round(PIPE[name]['test_auc'], 4),
        'Overfit Gap': round(PIPE[name]['gap'], 4),
        'Log Loss':    round(PIPE[name]['logloss'], 4),
        'Brier Score': round(PIPE[name]['brier'], 4),
        'Precision':   round(PIPE[name]['precision_final'], 3),
        'Recall':      round(PIPE[name]['recall_final'], 3),
        'F1':          round(PIPE[name]['f1_final'], 3),
    })
df_compare = pd.DataFrame(rows).set_index('Model')
print('Cross-Model Comparison (OOF-tuned thresholds):')
print(df_compare.to_string())

palette = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860']
c_map   = {name: c for name, c in zip(MODEL_NAMES, palette)}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Cross-Model Comparison -- OOF-Tuned Thresholds', fontsize=14, fontweight='bold')

w       = 0.13
offsets = np.arange(len(MODEL_NAMES)) - (len(MODEL_NAMES) - 1) / 2
metrics_h = ['Test AUC', 'Precision', 'Recall', 'F1']
metrics_l = ['Overfit Gap', 'Log Loss', 'Brier Score']
x_h = np.arange(len(metrics_h))
x_l = np.arange(len(metrics_l))

for i, name in enumerate(MODEL_NAMES):
    vals_h = [df_compare.loc[name, m] for m in metrics_h]
    vals_l = [df_compare.loc[name, m] for m in metrics_l]
    for ax, vals, xs in [(axes[0], vals_h, x_h), (axes[1], vals_l, x_l)]:
        bars = ax.bar(xs + offsets[i] * w, vals, w, label=name, color=c_map[name], alpha=0.85)
        for k, v in enumerate(vals):
            ax.text(xs[k] + offsets[i] * w, v + 0.003, f'{v:.3f}',
                    ha='center', va='bottom', fontsize=6)

axes[0].set_xticks(x_h); axes[0].set_xticklabels(metrics_h)
axes[0].set_ylim(0, 1.05); axes[0].set_title('Higher is Better')
axes[0].legend(fontsize=8); axes[0].grid(axis='y', alpha=0.3)

axes[1].set_xticks(x_l); axes[1].set_xticklabels(metrics_l)
axes[1].set_ylim(0, 0.85); axes[1].set_title('Lower is Better')
axes[1].legend(fontsize=8); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# Overlaying all 6 lift curves on one plot allows direct visual comparison of
# how quickly each model concentrates real hitmakers at the top of its ranking.
# A steeper initial curve = the model is better at prioritising true hitmakers.
# Also prints lift at key screening percentiles (10%, 20%, 30%, 50%) for a
# concrete, stakeholder-friendly summary: 'Model X finds 2.4x more hitmakers
# than random when screening the top 20% of artists.'

fig, ax = plt.subplots(figsize=(10, 7))
for name in MODEL_NAMES:
    y_p = PIPE[name]['y_proba_te']
    pop_frac, gain, _ = compute_lift(y_test.values, y_p)
    ax.plot(pop_frac * 100, gain * 100, lw=2, label=name, color=c_map[name])
ax.plot([0, 100], [0, 100], 'k--', lw=1.5, label='Random baseline')
ax.set_xlabel('% of Artists Screened (ranked by predicted probability)', fontsize=12)
ax.set_ylabel('% of Hitmakers Captured', fontsize=12)
ax.set_title('Cumulative Gain (Lift) -- All Models', fontsize=13)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100); ax.set_ylim(0, 105)
plt.tight_layout(); plt.show()

print(f'\n{"Model":<22}  {"@10%":>8}  {"@20%":>8}  {"@30%":>8}  {"@50%":>8}')
print('-' * 60)
for name in MODEL_NAMES:
    y_p = PIPE[name]['y_proba_te']
    pop_frac, gain, lift = compute_lift(y_test.values, y_p)
    lifts = []
    for pct in [0.10, 0.20, 0.30, 0.50]:
        idx = min(np.searchsorted(pop_frac, pct), len(lift) - 1)
        lifts.append(lift[idx])
    print(f'{name:<22}  {lifts[0]:>7.2f}x  {lifts[1]:>7.2f}x  {lifts[2]:>7.2f}x  {lifts[3]:>7.2f}x')

In [ ]:
# Cross-model prediction table showing which test artists the models disagree on most.
# Disagreement = standard deviation of predicted probabilities across all 6 models.
# High-disagreement artists where models split (some predict hitmaker, others don't)
# are the most informative cases for understanding model differences.
# These will be the focus of the SHAP waterfall analysis in the explainability
# notebook (ml_sandbox_22), run after reviewing these results.

short_names = [n[:12] for n in MODEL_NAMES]
all_probas  = {short: PIPE[name]['y_proba_te']
               for short, name in zip(short_names, MODEL_NAMES)}

df_pred_compare = pd.DataFrame(all_probas, index=y_test.index).round(3)
df_pred_compare['True Label']   = y_test.values
df_pred_compare['True Class']   = y_test.map({0.0: 'One-hit Wonder', 1.0: 'Hitmaker'}).values
df_pred_compare['Disagreement'] = df_pred_compare[short_names].std(axis=1).round(3)
df_pred_compare = df_pred_compare.sort_values('Disagreement', ascending=False)

print('Top 20 most disagreed-upon test artists (candidates for SHAP waterfall analysis):')
print(df_pred_compare.head(20).to_string())
print('\nBottom 10 most agreed-upon:')
print(df_pred_compare.tail(10).to_string())

In [ ]:
# Normalized permutation importance (sum=1 per model column) for cross-model
# feature comparison. Raw scores are on different scales across models, so
# normalization is needed before visual comparison.
# Grey = feature not selected by that model in its n_optimal feature set.
# Features sorted by mean normalized importance across models that selected them.

all_feats = sorted(set(
    feat for name in MODEL_NAMES
    for feat in PIPE[name]['perm_final']['Feature'].tolist()
))
df_imp = pd.DataFrame(index=all_feats, columns=MODEL_NAMES, dtype=float)
for name in MODEL_NAMES:
    imp = PIPE[name]['perm_final'].set_index('Feature')['Importance']
    for feat in all_feats:
        df_imp.loc[feat, name] = imp.get(feat, np.nan)

df_clipped = df_imp.clip(lower=0)
df_norm    = df_clipped.div(df_clipped.sum(axis=0), axis=1)
df_norm['_mean'] = df_norm.mean(axis=1)
df_norm = df_norm.sort_values('_mean', ascending=False).drop(columns='_mean')

fig, ax = plt.subplots(figsize=(13, max(6, len(all_feats) * 0.45)))
mask = df_norm.isna()
sns.heatmap(df_norm.astype(float), annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, mask=mask,
            cbar_kws={'label': 'Normalized importance (sum=1 per model)'},
            vmin=0, vmax=1, annot_kws={'size': 8})
for (i, j) in zip(*np.where(df_norm.isna().values)):
    ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=True, color='lightgrey', zorder=3))
    ax.text(j + 0.5, i + 0.5, 'n/s', ha='center', va='center', fontsize=8, color='gray')
ax.set_title(
    'Cross-Model Feature Importance (normalized, sum=1 per model)\nGrey = not selected',
    fontsize=12,
)
ax.set_xlabel('Model'); ax.set_ylabel('Feature')
plt.tight_layout(); plt.show()

In [ ]:
# Signed importance = permutation importance * sign(corr(feature, target)).
# Positive (red) = feature pushes toward Hitmaker.
# Negative (blue) = feature pushes toward One-hit Wonder.
# Normalized by max absolute value per model so both directions are comparable.
# Features that flip direction across models (one red, one blue) signal that
# the relationship is non-linear or context-dependent -- worth investigating
# in the SHAP waterfall analysis.

df_signed = pd.DataFrame(index=all_feats, columns=MODEL_NAMES, dtype=float)
for name in MODEL_NAMES:
    X_tr = PIPE[name]['X_train_final']
    imp  = PIPE[name]['perm_final'].set_index('Feature')['Importance']
    for feat in all_feats:
        if feat in X_tr.columns and feat in imp.index:
            corr = np.corrcoef(X_tr[feat].values, y_train.values)[0, 1]
            df_signed.loc[feat, name] = imp[feat] * np.sign(corr)

abs_max        = df_signed.abs().max(axis=0)
df_signed_norm = df_signed.div(abs_max, axis=1)
df_signed_norm = df_signed_norm.loc[df_norm.index]   # same row order as unsigned

fig, ax = plt.subplots(figsize=(13, max(6, len(all_feats) * 0.45)))
mask_s = df_signed_norm.isna()
sns.heatmap(df_signed_norm.astype(float), annot=True, fmt='.2f', cmap='RdBu_r',
            linewidths=0.5, ax=ax, mask=mask_s,
            cbar_kws={'label': 'Signed importance (red=->Hitmaker, blue=->One-hit Wonder)'},
            vmin=-1, vmax=1, annot_kws={'size': 8})
for (i, j) in zip(*np.where(df_signed_norm.isna().values)):
    ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=True, color='lightgrey', zorder=3))
    ax.text(j + 0.5, i + 0.5, 'n/s', ha='center', va='center', fontsize=8, color='gray')
ax.set_title(
    'Signed Feature Importance (direction x magnitude, normalized)\n'
    'Red = -> Hitmaker  .  Blue = -> One-hit Wonder  .  Grey = not selected',
    fontsize=12,
)
ax.set_xlabel('Model'); ax.set_ylabel('Feature')
plt.tight_layout(); plt.show()